# V4 CUB Consensus Repair

                Optional but recommended. This notebook does not retrain CUB models. It reruns the four older CUB XAI
                jobs with the updated consensus code so the CUB consensus table has all six model rows.

In [ ]:
%pip install -q timm pydicom captum grad-cam scikit-image scipy seaborn

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

PY = sys.executable
WORK = Path("/kaggle/working")

def run(cmd):
    cmd = [str(x) for x in cmd]
    print("\n$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

def input_roots():
    roots = [WORK]
    root = Path("/kaggle/input")
    if root.exists():
        roots += [p for p in root.iterdir() if p.is_dir()]
        datasets = root / "datasets"
        if datasets.exists():
            for owner in datasets.iterdir():
                if owner.is_dir():
                    roots += [p for p in owner.iterdir() if p.is_dir()]
    return roots

def find_code_source():
    for r in input_roots() + [Path.cwd()]:
        for p in [r, r / "spinexnet-code", *r.glob("**/spinexnet-code")]:
            if (p / "cub_200_generalization" / "run_cub_xai.py").exists():
                return p
    raise FileNotFoundError("Attach the updated spinexnet-code package/dataset.")

def first_existing(candidates, label, required=True):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    if required:
        raise FileNotFoundError(f"Could not find {label}")
    return None

SRC = find_code_source()
CODE = WORK / "spinexnet-code"
if SRC.resolve() != CODE.resolve():
    shutil.copytree(SRC, CODE, dirs_exist_ok=True)
CUB_CODE = CODE / "cub_200_generalization"

CUB_ROOT = first_existing(
    [r / "CUB_200_2011" for r in input_roots()] + [p for r in input_roots() for p in r.glob("**/CUB_200_2011")],
    "CUB_200_2011 dataset",
)
ORIGINAL_CUB_RESULT_ROOT = first_existing(
    [r / "cuba 200" / "dataset_no_npy" for r in input_roots()]
    + [r / "cub_200" / "dataset_no_npy" for r in input_roots()]
    + [r / "v4 results" / "cuba 200" / "dataset_no_npy" for r in input_roots()]
    + [
        r for r in input_roots()
        if (r / "cub_xai").exists() or (r / "cub_eval").exists() or (r / "cub_aggregate").exists()
    ],
    "original V4 CUB result root",
    required=False,
)
REPAIR_ROOT = WORK / "v4_repair_cub_consensus"
REPAIR_XAI_ROOT = REPAIR_ROOT / "cub_xai"
AGG_ROOT = REPAIR_ROOT / "cub_aggregate"
REPAIR_XAI_ROOT.mkdir(parents=True, exist_ok=True)
AGG_ROOT.mkdir(parents=True, exist_ok=True)

MODELS_TO_REPAIR = ["resnet50", "densenet121", "efficientnet_b4", "vit_small"]
FOLDS = [0, 1, 2, 3, 4]
MAX_SAMPLES_XAI = 300
FAITHFULNESS_STEPS = 20
GRADIENT_BASELINE_MODE = "mean"
FAITHFULNESS_BASELINE_MODE = "mean"

def find_checkpoint(model, fold):
    rels = [
        Path("cub_outputs") / model / f"fold_{fold}" / "best.pt",
        Path(model) / f"fold_{fold}" / "best.pt",
        Path("checkpoints") / f"{model}_fold_{fold}_best.pt",
    ]
    candidates = []
    for root in input_roots():
        candidates += [root / rel for rel in rels]
    candidates += [WORK / rel for rel in rels]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find CUB checkpoint for {model} fold {fold}")

print("CODE:", CODE)
print("CUB_ROOT:", CUB_ROOT)
print("ORIGINAL_CUB_RESULT_ROOT:", ORIGINAL_CUB_RESULT_ROOT)
print("REPAIR_ROOT:", REPAIR_ROOT)
for model in MODELS_TO_REPAIR:
    print(model, "fold_0 checkpoint:", find_checkpoint(model, 0))

## Rerun CUB XAI With Consensus

In [ ]:
METHODS = ["gradcam", "gradcam++", "integrated_gradients", "gradient_shap", "occlusion", "guided_backprop"]

for model in MODELS_TO_REPAIR:
    for fold in FOLDS:
        out = REPAIR_XAI_ROOT / f"fold_{fold}" / model
        out.mkdir(parents=True, exist_ok=True)
        cmd = [
            PY, CUB_CODE / "run_cub_xai.py",
            "--model", model,
            "--checkpoint", find_checkpoint(model, fold),
            "--cub-root", CUB_ROOT,
            "--fold", fold,
            "--output-dir", out,
            "--max-samples", MAX_SAMPLES_XAI,
            "--methods", *METHODS,
            "--faithfulness-steps", FAITHFULNESS_STEPS,
            "--consensus-every", 10,
            "--topk-consensus-methods", 3,
            "--gradient-baseline-mode", GRADIENT_BASELINE_MODE,
            "--faithfulness-baseline-mode", FAITHFULNESS_BASELINE_MODE,
        ]
        if model in {"vit_small", "deit_small"}:
            cmd += ["--enable-attention-rollout"]
        run(cmd)

## Aggregate Original + Repaired CUB Outputs

In [ ]:
input_roots_for_agg = [REPAIR_ROOT]
if ORIGINAL_CUB_RESULT_ROOT and ORIGINAL_CUB_RESULT_ROOT.exists():
    input_roots_for_agg.insert(0, ORIGINAL_CUB_RESULT_ROOT)

run([
    PY, CUB_CODE / "aggregate_cub_results.py",
    "--input-roots", *input_roots_for_agg,
    "--copy-inputs",
    "--output-dir", AGG_ROOT,
])

print("CUB repair outputs:")
print("  XAI:", REPAIR_XAI_ROOT)
print("  Aggregate:", AGG_ROOT)